# Notebook 08: breaking the best detector, part 2: contextual anomalies (daily series)

**Class 03_03, part 2 of 3.** A **contextual** anomaly is a value that is normal in itself and wrong *at that
moment*: 80 requests per second at noon is normal, at 3 a.m. it is not. The detectors of notebook 05 learn the
context (the daily cycle) from the first eight days of a series and score every step against it.

1. **Pick** the best detector on the four NAB series of notebook 05.
2. **Break** it: both detectors react to *change*, so a slow ramp hides an offset from them; and a *replay* of
   yesterday hides a real anomaly completely.
3. **Defend**: accumulate the evidence (CUSUM on the signed seasonal difference) and let several detectors vote.

**Threat model.** The attacker can add an offset to the reading (a miscalibrated or spoofed sensor); it grows
linearly over $R$ days or jumps at once, or replays an old day over the real one; the defender's rule is fixed in
advance (the NAB protocol: the threshold is the maximum score of one clean day, plus 10%). The size of the offset is
measured in standard deviations of the clean series; a detector *sees* the offset when it raises at least 12 alarm
steps (one hour) more than on the clean series.

In [ ]:
import os
import sys

os.environ.setdefault("KERAS_BACKEND", "jax")
import numpy as np
import pandas as pd
import polars as pl

sys.path[:0] = [".", "../../src"]  # this folder and the generic libraries of the repository
import aasplot as ap
import anomlib as an
import deeplib as dl
import robustlib as rl
from aasplot import BLUE, GOLD, GREEN, RED
from autoenc import build_autoencoder

an.table_config()
ap.setup("03")
D, W = dl.DAY, dl.DAY  # one day = 288 samples = the window length
SERIES = ["jumpsup", "jumpsdown", "nojump", "flatmiddle"]
CONV = dict(
    kind="conv", widths=(32, 16), code=None, activation="relu"
)  # the body of notebook 05, built by build_autoencoder


def fit(x, stride=1, seed=0):
    """A convolutional AE over one-day windows of the (standardised) series x."""
    return dl.fit_window_ae(build_autoencoder((W, 1), **CONV), dl.windows(x, W)[::stride], epochs=30, seed=seed)


def score(model, x):
    """Per-step score: squared error at t, averaged over the windows that contain t."""
    return dl.window_to_point(dl.window_error(model, dl.windows(x, W)), W, len(x))


def seasonal_diff(x):
    return np.r_[np.full(D, np.nan), np.abs(x[D:] - x[:-D])]


def longest_run(mask) -> int:
    best = run = 0
    for v in mask:
        run = run + 1 if v else 0
        best = max(best, run)
    return best

## A. Pick the best detector

Four detectors (three one-liners and the convolutional AE) on the four series; per cell: whether the anomaly is
found and how many false alarms the series raises. The best finds the most anomalies, then raises the fewest false
alarms.

In [ ]:
ONE = {"|x|": lambda x: np.abs(x),
       "seasonal diff": seasonal_diff,
       "moving std 1h": lambda x: pd.Series(x).rolling(12, min_periods=2).std().to_numpy()}  # fmt: skip
rows, curves = [], {}
for name in SERIES:
    _, te = dl.load_nab(name)
    v, y = te.value.to_numpy(), te.y.to_numpy()
    mu, sd = v[: 8 * D].mean(), v[: 8 * D].std()
    x = (v - mu) / sd
    m = fit(x[: 8 * D])
    fs = dict(ONE)
    fs["conv AE"] = lambda x, m=m: score(m, x)
    for k, f in fs.items():
        s = f(x)
        t = np.nanmax(s[8 * D : 9 * D]) * 1.1
        rows.append({"series": name, "detector": k, **dl.series_report(np.nan_to_num(s), y, t)})
        curves[(name, k)] = (s, t)
res = pl.DataFrame(rows)
print(res.pivot(on="detector", index="series", values="found"))
print(res.pivot(on="detector", index="series", values="false_alarms"))
print(res.pivot(on="detector", index="series", values="delay"))
summary = (res.group_by("detector").agg(pl.col("found").sum().alias("found of 4"), pl.col("false_alarms").sum().alias("false alarms"))
           .sort(["found of 4", "false alarms"], descending=[True, False]))  # fmt: skip
BEST = summary["detector"][0]
print(summary)
print("best:", BEST)

## B. Break it (1): a ramp hides an offset

The attacker adds an offset $\Delta$ to the reading from day 10 on, either at once ($R=0$) or growing linearly over
$R=1$ or $3$ days. The detectors stay as they were trained (days 1-8 of a clean series). A third detector is
added for the defence: **CUSUM on the signed seasonal difference** $d_t=x_t-x_{t-288}$, which adds up the
day-over-day change instead of comparing it with a limit. For each size we count the alarm steps in days 10-14; the
**blind spot** is the largest $\Delta$ whose count stays within 12 of the clean count.

In [ ]:
x0 = dl.load_nab("jumpsup")[0].value.to_numpy()  # the clean training series of the NAB protocol
x = (x0 - x0[: 8 * D].mean()) / x0[: 8 * D].std()
ae0 = fit(x[: 8 * D])
t_ae = score(ae0, x)[8 * D : 9 * D].max() * 1.1
t_diff = np.nanmax(seasonal_diff(x)[D : 9 * D]) * 1.1


def signed_cusum(x, k=0.5):
    """CUSUM of the day-over-day change d_t = x_t - x_{t-D}, scaled by its sd on the clean days 2-9."""
    d = np.r_[np.zeros(D), x[D:] - x[:-D]]
    return rl.cusum(d, d[D : 9 * D].mean(), d[D : 9 * D].std(), k)


h_cusum = signed_cusum(x)[D : 9 * D].max() * 1.1
DET = {
    "conv AE": (lambda v: score(ae0, v), t_ae),
    "seasonal diff": (seasonal_diff, t_diff),
    "CUSUM of the change": (signed_cusum, h_cusum),
}


def ramp(x, delta, R, start=9 * D):
    """x with an offset that grows linearly over R days from `start` and then stays at `delta` (R=0: a step)."""
    xa = x.copy()
    n = len(x) - start
    xa[start:] += delta * (np.minimum(1, np.arange(n) / (R * D)) if R else 1.0)
    return xa


def alarm_steps(name, v):
    f, t = DET[name]
    return int(np.nansum(f(v)[9 * D :] > t))


base = {k: alarm_steps(k, x) for k in DET}
DELTAS = [0.25, 0.5, 1, 1.5, 2, 3, 4]
rows = []
for R in [0, 1, 3]:
    for delta in DELTAS:
        xa = ramp(x, delta, R)
        rows.append({"ramp days": R, "offset (sd)": delta, **{k: alarm_steps(k, xa) - base[k] for k in DET}})
grid = pl.DataFrame(rows)
blind = {(R, det): max([d for d in DELTAS if grid.filter((pl.col("ramp days") == R) & (pl.col("offset (sd)") == d))[det][0] < 12], default=0)
         for R in (0, 1, 3) for det in DET}  # fmt: skip
print("clean alarm steps in days 10-14:", base)
print(grid.filter(pl.col("ramp days") == 3))
print("blind spot (largest offset in sd that raises fewer than 12 extra alarm steps):", blind)

In [ ]:
def blind_spots(blind) -> None:
    """Largest offset (in sd) that raises no hour-long alarm, for a step and for ramps over 1 and 3 days."""
    f, ax = ap.fig("full", 1.5)
    dets = [("conv AE", BLUE), ("seasonal diff", GOLD), ("CUSUM of the change", GREEN)]
    ap.grouped_bars(ax, ["step", "ramp over 1 day", "ramp over 3 days"], [(d, [blind[(R, d)] for R in (0, 1, 3)]) for d, _ in dets],
                    "largest hidden offset (sd)", colors=[c for _, c in dets], ymax=0.8)  # fmt: skip
    ax.legend(loc="upper left", ncol=3, fontsize=6.3)
    ap.save(f, "blind_spots")


blind_spots(blind)

**Reading.** The conv AE is the best detector of the four (it finds all four series with 39 false alarms; the seasonal
difference finds them too but echoes a day later, 689 false alarms). Both react to *change*: for a step the blind
spot is 0.25 sd, and a ramp over three days raises it to 0.5 sd for both (about a fifth of the daily swing of 2.5 sd)
because a ramp has no edge for the AE and a day-over-day difference far below the limit for the seasonal
difference. The offsets are small, and that is the point: this is the size of the lie that the best detector
tolerates for days. **CUSUM of the change** adds up the same day-over-day differences and sees every size of
the grid (blind spot below 0.25 sd), but its false alarms are the price (section C).

## C. Break it (2): replay yesterday

On `jumpsup` the anomaly is a day-long jump; both detectors find it. An attacker who controls the feed replays the
previous (normal) day over the anomaly window. Every value is a real normal value and the daily shape is intact:
the detectors check *consistency with the context*, not *truth*, so nothing is left to find.

In [ ]:
_, te = dl.load_nab("jumpsup")
v = te.value.to_numpy()
y_t = te.y.to_numpy()
xt = (v - x0[: 8 * D].mean()) / x0[: 8 * D].std()
i0, i1 = np.flatnonzero(y_t)[[0, -1]]
xr = xt.copy()
i0, i1 = i0 - D // 2, i1 + D // 2  # a margin: the anomaly itself is inside the label window
xr[i0 : i1 + 1] = xt[i0 - D : i1 + 1 - D]  # yesterday, over the anomaly
rows = []
for label, series in (("anomaly", xt), ("anomaly replayed over", xr)):
    for k, (f, t) in DET.items():
        s = np.nan_to_num(f(series))
        rows.append({"series": label, "detector": k, **dl.series_report(s, y_t, t)})
print(pl.DataFrame(rows))

In [ ]:
def nab_replay(xt, xr, y) -> None:
    """The jumpsup series with the anomaly (shaded) and with yesterday replayed over it."""
    f, axs = ap.fig("full", 1.3, nrows=2, sharex=True, gridspec_kw={"hspace": 0.35})
    for ax, series, title in ((axs[0], xt, "real anomaly"), (axs[1], xr, "yesterday replayed over it")):
        ax.plot(series, lw=0.5, color=BLUE)
        ax.fill_between(np.arange(len(series)), series.min(), series.max(), where=y == 1, color=RED, alpha=0.25, lw=0)
        ax.set_title(title, fontsize=7.5, loc="left", pad=1)
        ax.set(yticks=[])
        ax.grid(False)
    axs[1].set_xlabel("time step (5 min)")
    ap.save(f, "nab_replay")


nab_replay(xt, xr, y_t)

**Reading.** The seasonal difference is blind to the replay by construction: yesterday minus the day before is
normal, so it finds nothing (found 0, no false alarms). The conv AE still alarms, but at the *seams*: the replay
starts and ends with a small discontinuity, and the 13 alarms are at those edges, not at the content (without the
replay it has none on this series). An attacker who blends the seams removes even that. CUSUM of the change
raises 213 alarm steps on the unreplayed series: the most sensitive rule is also the noisiest one.

## D. Defend it: accumulate and vote

Against the ramp, accumulation (CUSUM of the change, section B) lowers the blind spot; against the replay nothing in
the series itself helps, only a **second source** of the same quantity (another sensor, a log, a person). Voting
between detectors that fail differently also helps: the blind spot of an OR rule is the smaller of the blind spots.

In [ ]:
rows = []
for R in (0, 1, 3):
    row = {"ramp days": R, **{k: blind[(R, k)] for k in DET}}
    row["OR of all three"] = min(blind[(R, k)] for k in DET)
    rows.append(row)
print(pl.DataFrame(rows))

**Reading.** Accumulation is the only defence that moves the blind spot: from 0.5 sd (AE, seasonal difference, 3-day ramp)
to below 0.25 sd (CUSUM of the change). Voting (OR) takes the smallest blind spot, at the sum of the false
alarms. Against the replay nothing in the series helps: the defence is a second source of the same quantity,
the same conclusion as the guard sensors of notebook 09.